# 实验二 房源采集与 MySQL 存储

## 第一阶段：Mac 本机环境与房源表检查

本机使用已有 MySQL 8.4，连接 `127.0.0.1:3306`，数据库为 `smartpark_lab`。教师提供的是 Windows Docker 脚本，本次使用 Mac 本机 MySQL，未运行这些脚本。建表 SQL 保存在 `01_mysql_stage1.sql`。第一阶段只检查环境和表结构，后续采集、CSV 与入库尚未进行。

In [1]:
import os
from getpass import getpass
import pymysql

password = os.environ.get("LAB02_MYSQL_PASSWORD")
if password is None:
    password = getpass("输入课程 MySQL student 密码：")

connection = pymysql.connect(
    host="127.0.0.1", port=3306, user="student",
    password=password, database="smartpark_lab",
    charset="utf8mb4", autocommit=False, connect_timeout=10
)
try:
    with connection.cursor() as cursor:
        cursor.execute("SELECT VERSION(), DATABASE()")
        version, database = cursor.fetchone()
        print("MySQL 版本：", version)
        print("当前数据库：", database)
        cursor.execute("SHOW TABLES")
        print("数据库中的表：", ", ".join(row[0] for row in cursor.fetchall()))
        cursor.execute("DESCRIBE houseinfo")
        print("\nhouseinfo 表结构：")
        for row in cursor.fetchall():
            print(" | ".join(str(value) for value in row))
        cursor.execute("SELECT COUNT(*) FROM houseinfo")
        print("\n入库前条数：", cursor.fetchone()[0])
finally:
    connection.close()
    print("数据库连接已关闭")

MySQL 版本： 8.4.11
当前数据库： smartpark_lab
数据库中的表： houseinfo, l05_books

houseinfo 表结构：
id | int | NO | PRI | None | auto_increment
title | varchar(200) | YES |  | None | 
house | varchar(50) | YES |  | None | 
address | varchar(200) | YES |  | None | 
struct | varchar(50) | YES |  | None | 
area | varchar(20) | YES |  | None | 
floor | varchar(20) | YES |  | None | 
maketime | varchar(20) | YES |  | None | 
price | varchar(20) | YES |  | None | 

入库前条数： 0
数据库连接已关闭


`id` 是数据库自动生成的主键；`title`、`house`、`address`、`struct`、`area`、`floor`、`maketime`、`price` 保存 8 个房源字段。此处查询结果为执行时的实际数据，不代表采集或入库已经完成。

## 第二、三阶段：完整在线数据与实际核对

普通 requests 收到访问验证后，改用 Firecrawl 正常取得教学站的10页 HTML，均已归档。下面回放已保存的真实响应，避免重复访问网站；在线获取流程在 `fetch_firecrawl.py`，原始请求清单在 `data/collection_summary.json`。

In [1]:
import json
from pathlib import Path
from bs4 import BeautifulSoup
from collect_houseinfo import FIELDS, parse_page, parse_house, save_csv

manifest = json.loads(Path("data/collection_summary.json").read_text())
print("在线采集完整：", manifest["complete"])
print("实际页数：", len(manifest["pages"]))
soup = BeautifulSoup(Path("html/online/page_01.html").read_text(), "html.parser")
rows = soup.select("div.property")
print("第一页房源节点：", len(rows))
print("首条八字段：", dict(zip(FIELDS, parse_house(rows[0]))))

在线采集完整： True
实际页数： 10
第一页房源节点： 60
首条八字段： {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}


先用 `select()` 找到每条房源，再用相对选择器 `select_one()` 取字段。面积根据单位定位，楼层按“层”定位，年代找“年建造”，缺失时填空。单价使用 `.property-price-average`；地址不额外插入空格。

In [2]:
records = []
for page in manifest["pages"]:
    html = Path("html/online", page["file"]).read_text(encoding="utf-8")
    page_records, info = parse_page(html, page["requested_url"])
    records.extend(page_records)
    print("页", page["page"], "本页", len(page_records), "累计", len(records))
    if page["page"] in (1, 2, 10):
        print("  href:", info["next_href"], "末页:", info["last_page"])
print("原始记录：", len(records))
print("八字段唯一记录：", len(set(records)))

页 1 本页 60 累计 60
  href: http://www.bspider.top/anjuke/?page=2 末页: False
页 2 本页 60 累计 120
  href: http://www.bspider.top/anjuke/?page=3 末页: False
页 3 本页 60 累计 180


页 4 本页 60 累计 240
页 5 本页 60 累计 300
页 6 本页 60 累计 360


页 7 本页 60 累计 420
页 8 本页 60 累计 480
页 9 本页 60 累计 540


页 10 本页 60 累计 600
  href: http://www.bspider.top/anjuke/?page=10# 末页: True
原始记录： 600
八字段唯一记录： 498


在线获取时用 `urljoin()` 补全下一页，检查教学站范围、重复 URL 和重复页面，两次请求间隔至少1秒。第10页没有 `next-active`，href 是本页地址加 `#`，说明下一页已禁用；Firecrawl 将原始相对锚点补成了完整地址。这里显示的是已保存响应的回放结果。

In [3]:
unique = save_csv(records, Path("data/houseinfo.csv"))
print("CSV 数据行数：", len(unique))
print("完全重复记录：", len(records)-len(unique))
for index in (0, 60, len(unique)-1):
    print("样本", index+1, dict(zip(FIELDS, unique[index])))

CSV 数据行数： 498
完全重复记录： 102
样本 1 {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}
样本 61 {'title': '万科物业自营 朝中庭 业主诚心出售', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '85.7㎡', 'floor': '低层(共27层)', 'maketime': '2018年建造', 'price': '16914元/㎡'}
样本 498 {'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}


CSV 按八字段完全相同去重，使用 `csv.writer`、`utf-8-sig`、`newline=""`，保存后用 `DictReader` 回读。第1、61、498条分别取自第1页、第2页和第10页。

In [4]:
def import_records(connection, records):
    """比较现有的 8 字段记录，插入不存在的记录并提交。"""
    cursor = connection.cursor()
    try:
        cursor.execute("SELECT COUNT(*) FROM houseinfo")
        before = cursor.fetchone()[0]
        cursor.execute("SELECT title, house, address, `struct`, area, floor, maketime, price FROM houseinfo")
        existing = set(cursor.fetchall())
        pending = []
        for record in records:
            if record not in existing:
                pending.append(record)
                existing.add(record)
        if pending:
            cursor.executemany(
                "INSERT INTO houseinfo (title, house, address, `struct`, area, floor, maketime, price) "
                "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)", pending
            )
        connection.commit()
        cursor.execute("SELECT COUNT(*) FROM houseinfo")
        return before, len(pending), cursor.fetchone()[0]
    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()


八个 `%s` 对应固定顺序的八个字段，房源文字作为参数传入。先查询已有记录，只插入未存在的组合；成功后 `commit()`，异常时 `rollback()`，无论结果如何都关闭游标。

In [5]:
import os
from getpass import getpass
import pymysql
from import_houseinfo import read_records

csv_records = read_records("data/houseinfo.csv")
password = os.environ.get("LAB02_MYSQL_PASSWORD") or getpass("MySQL 密码：")
connection = pymysql.connect(host="127.0.0.1", port=3306, user="student", password=password,
                             database="smartpark_lab", charset="utf8mb4", autocommit=False)
try:
    before, added, after = import_records(connection, csv_records)
    print("本次入库前：", before, "新增：", added, "入库后：", after)
    with connection.cursor() as cursor:
        cursor.execute("SELECT title, house, address, `struct`, area, floor, maketime, price FROM houseinfo")
        stored = list(cursor.fetchall())
    assert len(stored) == len(csv_records) and set(stored) == set(csv_records)
    print("全部八字段核对一致：", len(stored), "条 × 8 字段")
finally:
    connection.close()

本次入库前： 498 新增： 0 入库后： 498
全部八字段核对一致： 498 条 × 8 字段


此前用户快照已入库236条，本次完整在线数据新增262条，总数498；关闭连接独立重跑新增0。以上单元再次运行时库中已有498条，所以新增也是0。数据库自增长ID只保证主键唯一，不能代替八字段防重复。历史检查记录见 `data/import_verification.json`。